# 전체 코드

In [ ]:
import pandas as pd
import math

# 원본 파일 읽기
corp_df = pd.read_csv("/content/corp_list.csv", encoding="utf-8")

chunk_size = 500
total_rows = len(corp_df)
num_chunks = math.ceil(total_rows / chunk_size)

for i in range(num_chunks):
    start = i * chunk_size
    end = start + chunk_size
    chunk_df = corp_df.iloc[start:end]

    file_name = f"/content/corp_part_{i+1}.csv"
    chunk_df.to_csv(file_name, index=False)

    print(f"{file_name} 저장 완료 ({len(chunk_df)}개)")

/content/corp_part_1.csv 저장 완료 (500개)
/content/corp_part_2.csv 저장 완료 (500개)
/content/corp_part_3.csv 저장 완료 (500개)
/content/corp_part_4.csv 저장 완료 (500개)
/content/corp_part_5.csv 저장 완료 (500개)
/content/corp_part_6.csv 저장 완료 (500개)
/content/corp_part_7.csv 저장 완료 (500개)
/content/corp_part_8.csv 저장 완료 (386개)


In [ ]:
# 확인 코드
import requests
import pandas as pd

API_KEY = "api_key"

corp_code = "00109693"  # 예시 (삼성전자)
year = 2023
fs_div = "CFS"  # 연결재무제표
REPORT_CODE = "11013"  # 1분기

url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

params = {
    'crtfc_key': API_KEY,
    'corp_code': corp_code,
    'bsns_year': str(year),
    'reprt_code': REPORT_CODE,
    'fs_div': fs_div
}

res = requests.get(url, params=params)
data = res.json()

print("상태코드:", data.get("status"))
print("메시지:", data.get("message"))

if data.get("status") == "000":
    df = pd.DataFrame(data["list"])
    print("\n데이터 일부:")
    print(df[["account_nm", "thstrm_amount"]].head(20))
else:
    print("데이터 없음")


상태코드: 000
메시지: 정상

데이터 일부:
             account_nm  thstrm_amount
0                  유동자산  2986392292690
1              현금및현금성자산   755605316236
2                단기금융상품   122706297855
3             매출채권및기타채권   676205597877
4               당기법인세자산     2112553888
5                  재고자산  1219505692307
6       당기손익-공정가치측정금융자산    22927048010
7   기타포괄손익-공정가치 측정 금융자산         245000
8                파생상품자산    42890255554
9                기타유동자산    97419299730
10               매각예정자산    47019986233
11                비유동자산  9292097645716
12               장기금융상품    22815187188
13          장기매출채권및기타채권    80702621315
14          관계기업및공동기업주식  2420901176962
15      당기손익-공정가치측정금융자산    30312856010
16    기타포괄손익-공정가치측정금융자산    88049527447
17                 유형자산  3719159175066
18                투자부동산  1086294873544
19                 무형자산  1474079403704


In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import glob

# 설정
API_KEY = "8da93fd85f008c82ba98b3ed1e1cc4a5a3e98a5e"
START_YEAR = 2015
END_YEAR = 2025
REPORT_CODE = "11014"  # 보고서 종류(번호) 추출


# 최종 컬럼
FIN_COLS = [
    "stock_code", "year",

    # 수익성
    "roa", "roe", "net_margin", "op_margin",

    # 성장성
    "sales_growth_rate", "asset_growth",
    "op_income_growth", "net_income_growth",

    # 안정성
    "debt_ratio", "debt_to_assets",
    "non_current_debt_ratio", "current_debt_ratio",
    "equity_ratio", "equity_multiplier",
    "short_term_debt_ratio",

    # 자본잠식
    "capital_impairment_ratio",
    "is_capital_impaired",
    "is_full_impairment",

    # 이익잉여금
    "retained_earnings_deficit",
    "retained_earnings",
    "retained_earnings_ratio",
    "retained_earnings_to_equity",

    # 운전자본
    "net_working_capital",
    "net_working_capital_ratio",

    # 활동성
    "asset_turnover_ratio",

    # 기타
    "capital_ratio",
    "current_ratio",
    "current_assets_ratio",
    "non_current_assets_ratio"
]

# 안전 나눗셈
def safe_div(a, b):
    return np.where((b == 0) | (pd.isna(b)), np.nan, a / b)

# 단일 연도 데이터 수집
def get_integrated_data(corp_code, year):

    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

    def fetch(fs_div):
        params = {
            'crtfc_key': API_KEY,
            'corp_code': corp_code,
            'bsns_year': str(year),
            'reprt_code': REPORT_CODE,
            'fs_div': fs_div
        }
        try:
            res = requests.get(url, params=params, timeout=10)
            data = res.json()
        except:
            return None

        if data.get('status') != '000':
            return None

        return pd.DataFrame(data['list'])

    df = fetch("CFS")
    if df is None:
        df = fetch("OFS")

    if df is None:
        return None

    acc = {
        'total_assets': np.nan,
        'total_liabilities': np.nan,
        'equity': np.nan,
        'capital': np.nan,
        'retained_earnings': np.nan,
        'current_assets': np.nan,
        'non_current_assets': np.nan,
        'current_liabilities': np.nan,
        'non_current_liabilities': np.nan,
        'sales': np.nan,
        'operating_income': np.nan,
        'net_income': np.nan,
    }

    def safe(x):
        try:
            return float(str(x).replace(',', ''))
        except:
            return np.nan

    revenue_keywords = ['매출액', '영업수익', '수익']
    net_income_keywords = ['당기순이익', '반기순이익', '분기순이익']

    for _, row in df.iterrows():
        name = str(row['account_nm'])
        val = safe(row['thstrm_amount'])

        if name == '자산총계':
            acc['total_assets'] = val
        elif name == '부채총계':
            acc['total_liabilities'] = val
        elif name == '자본총계':
            acc['equity'] = val
        elif name == '자본금':
            acc['capital'] = val
        elif name == '이익잉여금':
            acc['retained_earnings'] = val
        elif name == '유동자산':
            acc['current_assets'] = val
        elif name == '비유동자산':
            acc['non_current_assets'] = val
        elif name == '유동부채':
            acc['current_liabilities'] = val
        elif name == '비유동부채':
            acc['non_current_liabilities'] = val
        elif any(k in name for k in revenue_keywords) and '이익' not in name:
            if pd.isna(acc['sales']):
                acc['sales'] = val
        elif '영업이익' in name:
            acc['operating_income'] = val
        elif any(k in name for k in net_income_keywords):
            acc['net_income'] = val

    return acc

# 기업 전체 기간 수집
def collect_company(corp_code, stock_code):

    records = []

    for year in range(START_YEAR, END_YEAR + 1):

        data = get_integrated_data(corp_code, year)

        if data is None:
            continue

        data["stock_code"] = stock_code.zfill(6)
        data["year"] = year

        records.append(data)
        time.sleep(0.15)

    if not records:
        return pd.DataFrame(columns=FIN_COLS)

    df = pd.DataFrame(records).sort_values("year")

    # 수익성
    df["roa"] = safe_div(df["net_income"], df["total_assets"])
    df["roe"] = safe_div(df["net_income"], df["equity"])
    df["net_margin"] = safe_div(df["net_income"], df["sales"])
    df["op_margin"] = safe_div(df["operating_income"], df["sales"])

    # 안정성
    df["debt_ratio"] = safe_div(df["total_liabilities"], df["equity"])
    df["debt_to_assets"] = safe_div(df["total_liabilities"], df["total_assets"])
    df["equity_ratio"] = safe_div(df["equity"], df["total_assets"])
    df["equity_multiplier"] = safe_div(df["total_assets"], df["equity"])
    df["current_ratio"] = safe_div(df["current_assets"], df["current_liabilities"])
    df["short_term_debt_ratio"] = safe_div(df["current_liabilities"], df["total_assets"])

    df["current_assets_ratio"] = safe_div(df["current_assets"], df["total_assets"])
    df["non_current_assets_ratio"] = safe_div(df["non_current_assets"], df["total_assets"])
    df["current_debt_ratio"] = safe_div(df["current_liabilities"], df["total_liabilities"])
    df["non_current_debt_ratio"] = safe_div(df["non_current_liabilities"], df["total_liabilities"])

    # 운전자본
    df["net_working_capital"] = df["current_assets"] - df["current_liabilities"]
    df["net_working_capital_ratio"] = safe_div(
        df["net_working_capital"], df["total_assets"]
    )

    # 활동성
    df["asset_turnover_ratio"] = safe_div(df["sales"], df["total_assets"])

    # 자본잠식
    df["capital_impairment_ratio"] = safe_div(df["equity"], df["capital"])
    df["is_capital_impaired"] = (df["equity"] < df["capital"]).astype(int)
    df["is_full_impairment"] = (df["equity"] <= 0).astype(int)

    # 이익잉여금
    df["retained_earnings_deficit"] = (df["retained_earnings"] < 0).astype(int)
    df["retained_earnings_ratio"] = safe_div(df["retained_earnings"], df["total_assets"])
    df["retained_earnings_to_equity"] = safe_div(df["retained_earnings"], df["equity"])

    # 기타
    df["capital_ratio"] = safe_div(df["capital"], df["total_assets"])

    # 성장성 (경고 제거 + 왜곡 방지)
    df["sales_growth_rate"] = df["sales"].pct_change(fill_method=None)
    df["asset_growth"] = df["total_assets"].pct_change(fill_method=None)
    df["op_income_growth"] = df["operating_income"].pct_change(fill_method=None)
    df["net_income_growth"] = df["net_income"].pct_change(fill_method=None)

    # 무한대 제거
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 성장률 극단값 제한
    growth_cols = [
        "sales_growth_rate",
        "asset_growth",
        "op_income_growth",
        "net_income_growth",
    ]
    df[growth_cols] = df[growth_cols].clip(-5, 5)

    # 비율 extreme 제한
    ratio_cols = [
        "roa","roe","net_margin","op_margin",
        "debt_ratio","debt_to_assets","equity_ratio",
        "equity_multiplier","current_ratio",
        "short_term_debt_ratio","current_assets_ratio",
        "non_current_assets_ratio","current_debt_ratio",
        "non_current_debt_ratio","net_working_capital_ratio",
        "asset_turnover_ratio","capital_impairment_ratio",
        "retained_earnings_ratio","retained_earnings_to_equity",
        "capital_ratio"
    ]

    df[ratio_cols] = df[ratio_cols].clip(-10, 10)

    # 종목 내부 결측 보정 (연도별 median)
    feature_cols = growth_cols + ratio_cols
    df[feature_cols] = df[feature_cols].apply(
        lambda x: x.fillna(x.median())
    )

    # 남은 NaN은 0 처리
    df[feature_cols] = df[feature_cols].fillna(0)

    return df.reindex(columns=FIN_COLS)




# 여러 파일 처리
file_list = sorted(glob.glob("/content/corp_part_*.xlsx"))

all_data = []

for file in file_list:

    print(f"\n===== {file} 처리 시작 =====")

    corp_df = pd.read_excel(file)

    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)
    corp_df["corp_code"] = corp_df["corp_code"].astype(str).str.zfill(8)

    for i, row in corp_df.iterrows():

        print(f"[{i+1}/{len(corp_df)}] {row['stock_code']}")

        df_result = collect_company(row["corp_code"], row["stock_code"])

        if not df_result.empty:
            all_data.append(df_result)

# 저장
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)

    final_df.to_excel(
        "/content/quarter3_financials.xlsx",
        index=False
    )

    print("\n엑셀 저장 완료")
else:
    print("\n수집된 데이터 없음")


===== /content/corp_part_4.xlsx 처리 시작 =====
[1/500] 054090
[2/500] 044380
[3/500] 271850
[4/500] 099440
[5/500] 033020
[6/500] 101360
[7/500] 071850
[8/500] 092790
[9/500] 419530
[10/500] 052960
[11/500] 900340
[12/500] 020400
[13/500] 260930
[14/500] 023590
[15/500] 311840
[16/500] 397500
[17/500] 367480
[18/500] 146060
[19/500] 115160
[20/500] 027390
[21/500] 001120
[22/500] 051380
[23/500] 033160
[24/500] 044450
[25/500] 079950
[26/500] 066570
[27/500] 096240
[28/500] 419120
[29/500] 210980
[30/500] 340930
[31/500] 263020
[32/500] 079190
[33/500] 377480
[34/500] 205100
[35/500] 339950
[36/500] 069140
[37/500] 026040
[38/500] 133750
[39/500] 330730
[40/500] 073110
[41/500] 440320
[42/500] 006800
[43/500] 242040
[44/500] 373170
[45/500] 033430
[46/500] 440110
[47/500] 010580
[48/500] 093640
[49/500] 005560
[50/500] 353060
[51/500] 067280
[52/500] 387310
[53/500] 039610
[54/500] 009770
[55/500] 466910
[56/500] 348210
[57/500] 340440
[58/500] 049130
[59/500] 464440
[60/500] 035250
[61/